# Road Accidents France — Exploratory Data Analysis

This notebook explores the cleaned BAAC dataset (2023) to answer key questions about road accident patterns in France.

**Questions explored:**
1. Are there peak hours for accidents?
2. Are nighttime accidents proportionally more severe?
3. How does injury severity vary by vehicle type?
4. How does injury severity vary by age group?
5. What is the geographic distribution of fatal accidents?
6. Are there correlations between variables?

**Data:** `accidents_france_2023_clean_for_EDA.csv` — output of `cleaning.ipynb`

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv('accidents_france_2023_clean_for_EDA.csv')
dftest = pd.read_csv('accidents_france_2024_clean_for_EDA.csv')
df_ml = pd.read_csv('accidents_france_2023_clean_for_ML.csv')

df['datetime'] = pd.to_datetime(df['datetime'])
dftest['datetime'] = pd.to_datetime(dftest['datetime'])

print(f"2023 dataset: {df.shape}")
print(f"2024 dataset: {dftest.shape}")
df.head()

## 1. Peak Hours for Accidents

**Question:** Are there specific hours of the day with higher accident frequency?

In [ ]:
df_by_hour = df.groupby(df['datetime'].dt.hour).size().reset_index(name='nb_accidents')
df_by_hour.columns = ['hour', 'nb_accidents']
df_by_hour['year'] = '2023'

df_by_hour_test = dftest.groupby(dftest['datetime'].dt.hour).size().reset_index(name='nb_accidents')
df_by_hour_test.columns = ['hour', 'nb_accidents']
df_by_hour_test['year'] = '2024'

df_combined = pd.concat([df_by_hour, df_by_hour_test])

px.line(df_combined, x='hour', y='nb_accidents', color='year',
        title='Number of accidents by hour of day — 2023 vs 2024', height=500)

**Observation:** Two clear peaks at 8am and 6pm, corresponding to commuting hours in France.

**Selection bias:** These peaks do not necessarily mean that driving at these hours is *more dangerous* — there is simply more traffic. To draw conclusions about actual risk, we would need to normalize by the volume of traffic at each hour.

**Further analysis:** Rather than counting accidents, we can look at the *proportion of fatal accidents* per hour to better assess risk.

## 2. Severity by Hour of Day

**Question:** Are nighttime accidents proportionally more severe?

In [ ]:
df['hour'] = df['datetime'].dt.hour
df_grav_hour = df.groupby('hour')['Injury_Severity'].value_counts(normalize=True).reset_index()

px.bar(df_grav_hour, x='hour', y='proportion', color='Injury_Severity',
       title='Injury severity proportion by hour of day', height=500)

**Observation:** The proportion of uninjured users is slightly lower at night (0h–5h), suggesting that nighttime accidents are proportionally more severe. The proportion of fatalities appears slightly higher at night.

**Possible explanations:** Higher speeds at night, driver fatigue, reduced visibility, delayed emergency response.

**Limitation:** Differences are small. A statistical significance test would be needed to confirm these differences are not due to chance.

## 3. Severity by Vehicle Type

**Question:** Do motorcyclists face higher fatality rates than car drivers?

In [ ]:
df_grav_veh = df.groupby('Vehicle_Category')['Injury_Severity'].value_counts(normalize=True).reset_index()

px.bar(df_grav_veh, x='Vehicle_Category', y='proportion',
       color='Injury_Severity', height=500,
       title='Injury severity by vehicle type')

## 4. Severity by Age Group

**Question:** Are elderly and young drivers more at risk of severe injuries?

In [ ]:
df['age_group'] = pd.cut(df['Age'],
                         bins=[0, 18, 25, 35, 45, 55, 65, 100],
                         labels=['<18', '18-25', '25-35', '35-45', '45-55', '55-65', '65+'])

df_grav_age = df.groupby('age_group')['Injury_Severity'].value_counts(normalize=True).reset_index()

px.bar(df_grav_age, x='age_group', y='proportion',
       color='Injury_Severity', height=500,
       title='Injury severity by age group')

## 5. Geographic Distribution of Fatal Accidents

**Question:** Are fatal accidents concentrated in specific regions?

In [ ]:
px.scatter(df, x='Longitude', y='Latitude', color='Injury_Severity',
           opacity=0.3, height=600,
           title='Geographic distribution of accidents by severity')

## 6. Correlation Matrix

**Question:** Are there linear relationships between numeric variables?

In [ ]:
df_num = df_ml.select_dtypes(include='number')
corr = df_num.corr()

px.imshow(corr, title='Correlation matrix (numeric features)', height=700)

**Observations:**
- `place` and `catu` are strongly correlated — the seat position depends on the user category (driver, passenger, pedestrian)
- `vma` and `agg` are negatively correlated — speed limits are lower in urban areas
- `vma` and `catr` are correlated — road type determines speed limits

**Regarding injury severity (`grav`):** All correlations with `grav` are weak, suggesting that no single variable alone can explain severity. This is consistent with ML results (see `ml.ipynb`).

**Limitation:** Pearson correlation measures linear relationships only. Since most variables are categorical encoded as integers, these correlations should be interpreted as indicative only. A chi-squared test would be more appropriate for categorical variables.